# Agent 4 — Gold KPI Validation Agent

Validates Gold KPIs for mathematical correctness,
cross-KPI consistency, anomaly detection, and range bounds.

Checks implemented:
  FORMULA   : KPI computed value matches defined formula
  CROSS_KPI : Related KPIs are logically consistent
  ANOMALY   : KPI value within historical baseline range
  RANGE     : KPI value within configured business bounds

Reads from:
  uc4_kpi_registry_b
  uc4_kpi_baseline_metrics_b
  uc2_gold_daily_store_metrics
  uc2_gold_daily_inventory_summary

Writes to:
  uc4_kpi_validation_run_b
  uc4_kpi_validation_results_b

## Cell 2: Parameters

Set business_date to any of the pilot dates:
  2026-06-19  Clean live batch         (expect all PASS)
  2026-06-22  Scenario A cross-KPI     (expect CRITICAL)
  2026-06-23  Scenario B anomaly       (expect CRITICAL)
  2026-06-24  Scenario C formula       (expect CRITICAL)

Or any historical date 2026-05-19 to 2026-06-17
for a clean historical validation.

In [0]:
dbutils.widgets.text(
    "business_date",
    "2026-06-19",
    "Business Date (YYYY-MM-DD)"
)

dbutils.widgets.text(
    "gold_table_filter",
    "",
    "Gold Table Filter (blank = all registered tables)"
)

from datetime import date, datetime, timedelta

BUSINESS_DATE_STR = dbutils.widgets.get("business_date").strip()
GOLD_TABLE_FILTER = dbutils.widgets.get("gold_table_filter").strip()

# ── Resolve business date ──────────────────────────────────────────────────
# Default to 2026-06-19 (clean live batch)
# rather than date.today() which points outside pilot data

if not BUSINESS_DATE_STR:
    BUSINESS_DATE = date(2026, 6, 19)
    date_source   = "default (pilot clean date)"
else:
    try:
        BUSINESS_DATE = date.fromisoformat(BUSINESS_DATE_STR)
        date_source   = "provided parameter"
    except ValueError:
        BUSINESS_DATE = date(2026, 6, 19)
        date_source   = "fallback (invalid format)"

# ── Validate date is within pilot data range ───────────────────────────────

HISTORICAL_START = date(2026, 5, 19)
HISTORICAL_END   = date(2026, 6, 17)

CONTROLLED_DATES = {
    date(2026, 6, 19): "clean live batch",
    date(2026, 6, 20): "Agent 1 volume drop",
    date(2026, 6, 21): "Agent 1 null spike",
    date(2026, 6, 22): "Agent 4 Scenario A — cross-KPI",
    date(2026, 6, 23): "Agent 4 Scenario B — anomaly",
    date(2026, 6, 24): "Agent 4 Scenario C — formula",
}

if HISTORICAL_START <= BUSINESS_DATE <= HISTORICAL_END:
    date_type = "historical clean"
elif BUSINESS_DATE in CONTROLLED_DATES:
    date_type = CONTROLLED_DATES[BUSINESS_DATE]
else:
    date_type = "WARNING: outside pilot data range — Gold tables may be empty"

RUN_START_TIME = datetime.now()

print("=" * 60)
print("AGENT 4 — GOLD KPI VALIDATION AGENT")
print("=" * 60)
print(f"Business date  : {BUSINESS_DATE}  ({date_source})")
print(f"Date type      : {date_type}")
print(f"Gold filter    : {GOLD_TABLE_FILTER or 'All registered tables'}")
print(f"Started at     : {RUN_START_TIME}")
print("=" * 60)

In [0]:
import uuid
import json
from datetime import date, datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, TimestampType,
    DoubleType, IntegerType, BooleanType
)
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ── Catalog and Schema ─────────────────────────────────────────────────────

dbutils.widgets.text("catalog", "learning_dev", "Target Catalog")
dbutils.widgets.text("schema", "s819534", "Target Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

# ── Table References ───────────────────────────────────────────────────────

KPI_REGISTRY           = f"{CATALOG}.{SCHEMA}.uc4_kpi_registry_b"
KPI_BASELINE_METRICS   = f"{CATALOG}.{SCHEMA}.uc4_kpi_baseline_metrics_b"
KPI_VALIDATION_RUN     = f"{CATALOG}.{SCHEMA}.uc4_kpi_validation_run_b"
KPI_VALIDATION_RESULTS = f"{CATALOG}.{SCHEMA}.uc4_kpi_validation_results_b"

# ── Fixed Baseline Date ────────────────────────────────────────────────────
# Must match the BASELINE_COMPUTATION_DATE used in
# 01_kpi_baseline_computation
# The load_baseline function uses this to find the correct baseline record

BASELINE_DATE = date(2026, 6, 25)

# ── Severity Levels ────────────────────────────────────────────────────────

STATUS_PASS     = "PASS"
STATUS_WARNING  = "WARNING"
STATUS_CRITICAL = "CRITICAL"
STATUS_ERROR    = "ERROR"

SEVERITY_RANK = {
    STATUS_PASS    : 0,
    STATUS_WARNING : 1,
    STATUS_CRITICAL: 2,
    STATUS_ERROR   : 3
}

# ── AI Confidence Threshold ────────────────────────────────────────────────
# Agent 4 triggers AI on any failure (even a single WARNING)
# because KPI issues always need explanation for business stakeholders

AI_MIN_FAILED_CHECKS  = 1
AI_MIN_CRITICAL_COUNT = 0

# ── Output Schemas ─────────────────────────────────────────────────────────

RUN_SCHEMA = StructType([
    StructField("validation_run_id",            StringType(),    False),
    StructField("gold_fully_qualified_name",     StringType(),    False),
    StructField("business_date",                DateType(),      False),
    StructField("triggered_by",                 StringType(),    True),
    StructField("agent2_reconciliation_run_id", StringType(),    True),
    StructField("validation_start_timestamp",   TimestampType(), False),
    StructField("validation_end_timestamp",     TimestampType(), True),
    StructField("validation_duration_seconds",  DoubleType(),    True),
    StructField("overall_status",               StringType(),    False),
    StructField("total_kpis_checked",           IntegerType(),   False),
    StructField("total_kpis_passed",            IntegerType(),   False),
    StructField("total_kpis_warned",            IntegerType(),   False),
    StructField("total_kpis_critical",          IntegerType(),   False),
    StructField("ai_enrichment_status",         StringType(),    False),
    StructField("ai_enrichment_timestamp",      TimestampType(), True),
    StructField("alert_sent",                   BooleanType(),   True),
    StructField("alert_sent_timestamp",         TimestampType(), True),
    StructField("alert_channel",                StringType(),    True),
    StructField("created_timestamp",            TimestampType(), False),
    StructField("created_by",                   StringType(),    False),
])

RESULTS_SCHEMA = StructType([
    StructField("result_id",                    StringType(),    False),
    StructField("validation_run_id",            StringType(),    False),
    StructField("gold_fully_qualified_name",    StringType(),    False),
    StructField("business_date",               DateType(),      False),
    StructField("kpi_name",                    StringType(),    False),
    StructField("kpi_column",                  StringType(),    False),
    StructField("check_type",                  StringType(),    False),
    StructField("check_status",                StringType(),    False),
    StructField("observed_value",              DoubleType(),    True),
    StructField("expected_value",              DoubleType(),    True),
    StructField("baseline_value",              DoubleType(),    True),
    StructField("deviation_percentage",        DoubleType(),    True),
    StructField("threshold_warning",           DoubleType(),    True),
    StructField("threshold_critical",          DoubleType(),    True),
    StructField("check_detail",               StringType(),    True),
    StructField("ai_summary",                 StringType(),    True),
    StructField("ai_recommended_action",       StringType(),    True),
    StructField("ai_failure_category",         StringType(),    True),
    StructField("ai_confidence_score",         DoubleType(),    True),
    StructField("ai_model_version",            StringType(),    True),
    StructField("ai_invocation_id",            StringType(),    True),
    StructField("created_timestamp",           TimestampType(), False),
    StructField("ai_enriched_timestamp",       TimestampType(), True),
])

print("Configuration loaded.")
print(f"Baseline date used for lookups: {BASELINE_DATE}")

## Check Implementations

In [0]:
def load_gold_data_for_date(
    gold_fqn      : str,
    biz_date_col  : str,
    business_date : date
):
    """Loads Gold table data for a specific business date."""
    df = spark.table(gold_fqn)
    if biz_date_col and biz_date_col in df.columns:
        return df.filter(
            F.col(biz_date_col) == F.lit(str(business_date))
        )
    return df


def load_baseline(
    gold_fqn  : str,
    kpi_name  : str,
    kpi_column: str
) -> dict:
    """
    Loads the baseline computed on BASELINE_DATE (2026-06-25)
    for a specific KPI.

    Uses a fixed baseline date rather than the most recent
    to ensure consistent behaviour regardless of when
    this notebook is run.
    """
    rows = (
        spark.table(KPI_BASELINE_METRICS)
             .filter(
                 (F.col("gold_fully_qualified_name") == gold_fqn) &
                 (F.col("kpi_name") == kpi_name) &
                 (F.col("kpi_column") == kpi_column) &
                 (F.col("baseline_date") == F.lit(str(BASELINE_DATE)))
             )
             .limit(1)
             .collect()
    )

    if not rows:
        logger.warning(
            f"No baseline found for {kpi_name} on {BASELINE_DATE}. "
            f"Run 01_kpi_baseline_computation first."
        )
        return None

    r = rows[0].asDict()
    return {
        "mean_value"  : r.get("mean_value"),
        "stddev_value": r.get("stddev_value"),
        "min_value"   : r.get("min_value"),
        "max_value"   : r.get("max_value"),
        "sample_count": r.get("sample_count"),
        "status"      : r.get("baseline_status")
    }


def build_result(
    run_id          : str,
    gold_fqn        : str,
    business_date   : date,
    kpi_name        : str,
    kpi_column      : str,
    check_type      : str,
    check_status    : str,
    observed_value  : float,
    expected_value  : float,
    baseline_value  : float,
    deviation_pct   : float,
    warn_threshold  : float,
    crit_threshold  : float,
    check_detail    : dict
) -> dict:
    """Builds a single KPI validation result record."""
    return {
        "result_id"                 : str(uuid.uuid4()),
        "validation_run_id"         : run_id,
        "gold_fully_qualified_name" : gold_fqn,
        "business_date"             : business_date,
        "kpi_name"                  : kpi_name,
        "kpi_column"                : kpi_column,
        "check_type"                : check_type,
        "check_status"              : check_status,
        "observed_value"            : observed_value,
        "expected_value"            : expected_value,
        "baseline_value"            : baseline_value,
        "deviation_percentage"      : deviation_pct,
        "threshold_warning"         : warn_threshold,
        "threshold_critical"        : crit_threshold,
        "check_detail"              : json.dumps(check_detail),
        "ai_summary"                : None,
        "ai_recommended_action"     : None,
        "ai_failure_category"       : None,
        "ai_confidence_score"       : None,
        "ai_model_version"          : None,
        "ai_invocation_id"          : None,
        "created_timestamp"         : datetime.now(),
        "ai_enriched_timestamp"     : None
    }


# ── Check 1: Formula Validation ────────────────────────────────────────────

def check_kpi_formula(
    gold_df       : object,
    kpi           : dict,
    run_id        : str,
    gold_fqn      : str,
    business_date : date
) -> dict:
    """
    Validates that a KPI column matches its defined formula.
    """
    t_start    = datetime.now()
    kpi_name   = kpi["kpi_name"]
    kpi_column = kpi["kpi_column"]
    formula    = kpi.get("formula_expression", "")

    print(f"    [FORMULA] {kpi_name}: {formula}")

    if not formula:
        print(f"      No formula defined. Skipping.")
        return None

    TOLERANCE = 0.01

    try:
        total_rows = gold_df.count()
        if total_rows == 0:
            return None

        if formula == "return_count / total_transactions":
            violation_df = gold_df.filter(
                F.abs(
                    F.col("return_rate") -
                    (F.col("return_count").cast("double") /
                     F.col("total_transactions").cast("double"))
                ) > TOLERANCE
            )
        elif formula == "total_net_amount / total_transactions":
            violation_df = gold_df.filter(
                F.abs(
                    F.col("avg_basket_value") -
                    (F.col("total_net_amount") /
                     F.col("total_transactions").cast("double"))
                ) > TOLERANCE
            )
        elif formula == "total_quantity_in - total_quantity_out":
            violation_df = gold_df.filter(
                F.abs(
                    F.col("net_quantity_change") -
                    (F.col("total_quantity_in") -
                     F.col("total_quantity_out"))
                ) > TOLERANCE
            )
        else:
            print(f"      Unknown formula pattern: {formula}. Skipping.")
            return None

        violation_count = violation_df.count()
        violation_rate  = violation_count / total_rows * 100

        status = STATUS_CRITICAL if violation_count > 0 else STATUS_PASS

        detail = {
            "formula"        : formula,
            "tolerance"      : TOLERANCE,
            "total_rows"     : total_rows,
            "violation_count": violation_count,
            "violation_rate" : round(violation_rate, 4)
        }

        print(
            f"      Violations: {violation_count}/{total_rows}  "
            f"Status: {status}"
        )

        return build_result(
            run_id, gold_fqn, business_date,
            kpi_name, kpi_column, "FORMULA",
            status,
            observed_value  = float(violation_count),
            expected_value  = 0.0,
            baseline_value  = 0.0,
            deviation_pct   = round(violation_rate, 4),
            warn_threshold  = 0.0,
            crit_threshold  = 0.0,
            check_detail    = detail
        )

    except Exception as e:
        print(f"      ERROR: {str(e)}")
        return None



# ── Check 2: Cross-KPI Consistency ─────────────────────────────────────────

def check_cross_kpi(
    gold_df       : object,
    kpi           : dict,
    run_id        : str,
    gold_fqn      : str,
    business_date : date
) -> dict:
    """
    Validates logical consistency between related KPIs.
    """
    t_start    = datetime.now()
    kpi_name   = kpi["kpi_name"]
    kpi_column = kpi["kpi_column"]
    formula    = kpi.get("formula_expression", "")

    print(f"    [CROSS_KPI] {kpi_name}: {formula}")

    if not formula:
        return None

    try:
        total_rows = gold_df.count()
        if total_rows == 0:
            return None

        if formula == "return_count <= total_transactions":
            violation_df = gold_df.filter(
                F.col("return_count") > F.col("total_transactions")
            )
        elif formula == "total_net_amount <= total_gross_amount":
            violation_df = gold_df.filter(
                F.col("total_net_amount") > F.col("total_gross_amount")
            )
        elif formula == "total_quantity_in >= 0":
            violation_df = gold_df.filter(
                F.col("total_quantity_in") < 0
            )
        elif formula == "return_rate <= 1.0":
            violation_df = gold_df.filter(
                F.col("return_rate") > 1.0
            )
        elif formula == "total_discount_amount <= total_gross_amount":
            violation_df = gold_df.filter(
                F.col("total_discount_amount") > F.col("total_gross_amount")
            )
        else:
            print(f"      Unknown cross-KPI rule: {formula}. Skipping.")
            return None

        violation_count = violation_df.count()
        violation_rate  = violation_count / total_rows * 100

        status = STATUS_CRITICAL if violation_count > 0 else STATUS_PASS

        detail = {
            "rule"           : formula,
            "total_rows"     : total_rows,
            "violation_count": violation_count,
            "violation_rate" : round(violation_rate, 4)
        }

        if violation_count > 0:
            sample = violation_df.limit(3).collect()
            detail["sample_violations"] = [
                {kpi_column: row.asDict().get(kpi_column)}
                for row in sample
            ]

        print(
            f"      Violations: {violation_count}/{total_rows}  "
            f"Status: {status}"
        )

        return build_result(
            run_id, gold_fqn, business_date,
            kpi_name, kpi_column, "CROSS_KPI",
            status,
            observed_value  = float(violation_count),
            expected_value  = 0.0,
            baseline_value  = 0.0,
            deviation_pct   = round(violation_rate, 4),
            warn_threshold  = 0.0,
            crit_threshold  = 0.0,
            check_detail    = detail
        )

    except Exception as e:
        print(f"      ERROR: {str(e)}")
        return None


# ── Check 3: KPI Anomaly Detection ─────────────────────────────────────────

# def check_kpi_anomaly(
#     gold_df       : object,
#     kpi           : dict,
#     baseline      : dict,
#     run_id        : str,
#     gold_fqn      : str,
#     business_date : date
# ) -> dict:
#     """
#     Detects when a KPI value deviates significantly from
#     its historical baseline mean.

#     Two-level check:
#       Level 1: Portfolio average vs baseline mean
#                (catches large-scale shifts like Scenario B)
#       Level 2: Count of individual rows exceeding threshold
#                (catches per-store/per-warehouse issues)
#     """
#     t_start    = datetime.now()
#     kpi_name   = kpi["kpi_name"]
#     kpi_column = kpi["kpi_column"]
#     warn_pct   = float(kpi.get("warning_deviation_pct") or 25.0)
#     crit_pct   = float(kpi.get("critical_deviation_pct") or 50.0)

#     print(f"    [ANOMALY] {kpi_name}")

#     if not baseline:
#         print(f"      No baseline available. Skipping.")
#         return None

#     baseline_mean = baseline.get("mean_value")
#     if baseline_mean is None or baseline_mean == 0:
#         print(f"      Baseline mean is zero or null. Skipping.")
#         return None

#     try:
#         total_rows = gold_df.count()
#         if total_rows == 0:
#             return None

#         # ── Level 1: Portfolio average vs baseline ─────────────────────────
#         current_avg = gold_df.agg(
#             F.avg(kpi_column).alias("avg")
#         ).collect()[0]["avg"]

#         if current_avg is None:
#             return None

#         portfolio_deviation = (
#             abs(current_avg - baseline_mean)
#             / abs(baseline_mean) * 100
#         )

#         # ── Level 2: Count of individual rows breaching threshold ──────────
#         warn_lower = baseline_mean * (1 - warn_pct / 100)
#         warn_upper = baseline_mean * (1 + warn_pct / 100)
#         crit_lower = baseline_mean * (1 - crit_pct / 100)
#         crit_upper = baseline_mean * (1 + crit_pct / 100)

#         rows_breach_warn = gold_df.filter(
#             (F.col(kpi_column) < warn_lower) |
#             (F.col(kpi_column) > warn_upper)
#         ).count()

#         rows_breach_crit = gold_df.filter(
#             (F.col(kpi_column) < crit_lower) |
#             (F.col(kpi_column) > crit_upper)
#         ).count()

#         warn_breach_pct = rows_breach_warn / total_rows * 100
#         crit_breach_pct = rows_breach_crit / total_rows * 100

#         # ── Status: worst of portfolio and row-level checks ────────────────
#         if portfolio_deviation >= crit_pct or crit_breach_pct >= 20:
#             status = STATUS_CRITICAL
#         elif portfolio_deviation >= warn_pct or warn_breach_pct >= 20:
#             status = STATUS_WARNING
#         else:
#             status = STATUS_PASS

#         detail = {
#             "kpi_column"             : kpi_column,
#             "portfolio_avg"          : round(float(current_avg), 4),
#             "baseline_mean"          : round(float(baseline_mean), 4),
#             "portfolio_deviation_pct": round(portfolio_deviation, 4),
#             "total_rows"             : total_rows,
#             "rows_breach_warning"    : int(rows_breach_warn),
#             "rows_breach_critical"   : int(rows_breach_crit),
#             "warn_breach_pct"        : round(warn_breach_pct, 2),
#             "crit_breach_pct"        : round(crit_breach_pct, 2),
#             "warn_zone"              : [
#                 round(warn_lower, 4), round(warn_upper, 4)
#             ],
#             "crit_zone"              : [
#                 round(crit_lower, 4), round(crit_upper, 4)
#             ],
#             "direction"              : (
#                 "DROP" if current_avg < baseline_mean else "SPIKE"
#             ),
#             "trigger"                : (
#                 "PORTFOLIO_AVG"
#                 if portfolio_deviation >= warn_pct
#                 else f"ROW_BREACH ({rows_breach_warn} rows)"
#             )
#         }

#         print(
#             f"      Portfolio avg   : {current_avg:.4f}  "
#             f"(baseline: {baseline_mean:.4f}, "
#             f"deviation: {portfolio_deviation:.2f}%)"
#         )
#         print(
#             f"      Rows > warn zone: {rows_breach_warn}/{total_rows} "
#             f"({warn_breach_pct:.1f}%)"
#         )
#         print(
#             f"      Rows > crit zone: {rows_breach_crit}/{total_rows} "
#             f"({crit_breach_pct:.1f}%)"
#         )
#         print(f"      Status          : {status}")

#         return build_result(
#             run_id, gold_fqn, business_date,
#             kpi_name, kpi_column, "ANOMALY",
#             status,
#             observed_value  = round(float(current_avg), 6),
#             expected_value  = round(float(baseline_mean), 6),
#             baseline_value  = round(float(baseline_mean), 6),
#             deviation_pct   = round(portfolio_deviation, 4),
#             warn_threshold  = warn_pct,
#             crit_threshold  = crit_pct,
#             check_detail    = detail
#         )

#     except Exception as e:
#         print(f"      ERROR: {str(e)}")
#         return None


def check_kpi_anomaly(
    gold_df       : object,
    kpi           : dict,
    baseline      : dict,
    run_id        : str,
    gold_fqn      : str,
    business_date : date
) -> dict:
    """
    Detects when a KPI value deviates significantly from
    its historical baseline mean.

    Uses portfolio-level comparison only:
      Compares today's average KPI value against the
      historical baseline mean and flags when deviation
      exceeds configured warning or critical thresholds.

    Design decision:
      Individual row-level breach counting was removed
      because Gold tables typically have few rows (25 stores,
      20 warehouse/event combinations) making row-level
      statistics unreliable and prone to false positives.
      Portfolio average is the correct signal for Gold KPIs.
    """
    t_start    = datetime.now()
    kpi_name   = kpi["kpi_name"]
    kpi_column = kpi["kpi_column"]
    warn_pct   = float(kpi.get("warning_deviation_pct") or 25.0)
    crit_pct   = float(kpi.get("critical_deviation_pct") or 50.0)

    print(f"    [ANOMALY] {kpi_name}")

    if not baseline:
        print(f"      No baseline available. Skipping.")
        return None

    baseline_mean = baseline.get("mean_value")
    if baseline_mean is None or baseline_mean == 0:
        print(f"      Baseline mean is zero or null. Skipping.")
        return None

    try:
        total_rows = gold_df.count()
        if total_rows == 0:
            return None

        # ── Portfolio average vs baseline mean ─────────────────────────────
        current_avg_row = gold_df.agg(
            F.avg(kpi_column).alias("avg")
        ).collect()[0]
        current_avg = current_avg_row["avg"]

        if current_avg is None:
            print(f"      No data for {kpi_column} on {business_date}")
            return None

        portfolio_deviation = (
            abs(current_avg - baseline_mean)
            / abs(baseline_mean) * 100
        )

        # ── Status from portfolio deviation only ───────────────────────────
        if portfolio_deviation >= crit_pct:
            status = STATUS_CRITICAL
        elif portfolio_deviation >= warn_pct:
            status = STATUS_WARNING
        else:
            status = STATUS_PASS

        detail = {
            "kpi_column"             : kpi_column,
            "portfolio_avg"          : round(float(current_avg), 4),
            "baseline_mean"          : round(float(baseline_mean), 4),
            "portfolio_deviation_pct": round(portfolio_deviation, 4),
            "total_rows"             : total_rows,
            "direction"              : (
                "DROP" if current_avg < baseline_mean else "SPIKE"
            ),
            "warn_threshold_pct"     : warn_pct,
            "crit_threshold_pct"     : crit_pct
        }

        print(
            f"      Portfolio avg   : {current_avg:.4f}  "
            f"(baseline: {baseline_mean:.4f})"
        )
        print(
            f"      Deviation       : {portfolio_deviation:.2f}%  "
            f"(warn {warn_pct}%  crit {crit_pct}%)"
        )
        print(f"      Status          : {status}")

        return build_result(
            run_id, gold_fqn, business_date,
            kpi_name, kpi_column, "ANOMALY",
            status,
            observed_value  = round(float(current_avg), 6),
            expected_value  = round(float(baseline_mean), 6),
            baseline_value  = round(float(baseline_mean), 6),
            deviation_pct   = round(portfolio_deviation, 4),
            warn_threshold  = warn_pct,
            crit_threshold  = crit_pct,
            check_detail    = detail
        )

    except Exception as e:
        print(f"      ERROR: {str(e)}")
        return None


def check_kpi_range(
    gold_df       : object,
    kpi           : dict,
    run_id        : str,
    gold_fqn      : str,
    business_date : date
) -> dict:
    """
    Validates that KPI values fall within configured business bounds.
    Counts rows where the KPI exceeds the configured min or max.
    """
    t_start    = datetime.now()
    kpi_name   = kpi["kpi_name"]
    kpi_column = kpi["kpi_column"]
    min_val    = kpi.get("min_value")
    max_val    = kpi.get("max_value")

    print(f"    [RANGE] {kpi_name}: [{min_val}, {max_val}]")

    if min_val is None and max_val is None:
        print(f"      No range configured. Skipping.")
        return None

    try:
        total_rows = gold_df.count()
        if total_rows == 0:
            return None

        violation_cond = F.lit(False)
        if min_val is not None:
            violation_cond = violation_cond | (
                F.col(kpi_column) < F.lit(min_val)
            )
        if max_val is not None:
            violation_cond = violation_cond | (
                F.col(kpi_column) > F.lit(max_val)
            )

        violation_count = gold_df.filter(
            F.col(kpi_column).isNotNull()
        ).filter(violation_cond).count()

        violation_rate = (
            violation_count / total_rows * 100
            if total_rows > 0 else 0.0
        )

        warn_pct = float(kpi.get("warning_deviation_pct") or 5.0)
        crit_pct = float(kpi.get("critical_deviation_pct") or 20.0)

        if violation_count > 0:
            if violation_rate >= crit_pct:
                status = STATUS_CRITICAL
            else:
                status = STATUS_WARNING
        else:
            status = STATUS_PASS

        detail = {
            "kpi_column"     : kpi_column,
            "configured_min" : min_val,
            "configured_max" : max_val,
            "total_rows"     : total_rows,
            "violation_count": violation_count,
            "violation_rate" : round(violation_rate, 4)
        }

        print(
            f"      Violations: {violation_count}/{total_rows}  "
            f"Status: {status}"
        )

        return build_result(
            run_id, gold_fqn, business_date,
            kpi_name, kpi_column, "RANGE",
            status,
            observed_value  = float(violation_count),
            expected_value  = 0.0,
            baseline_value  = 0.0,
            deviation_pct   = round(violation_rate, 4),
            warn_threshold  = warn_pct,
            crit_threshold  = crit_pct,
            check_detail    = detail
        )

    except Exception as e:
        print(f"      ERROR: {str(e)}")
        return None

print("All check functions loaded.")

## Write Functions

In [0]:
def write_validation_results(results: list) -> bool:
    if not results:
        return True
    col_order = [f.name for f in RESULTS_SCHEMA.fields]
    rows      = [tuple(r.get(col) for col in col_order) for r in results]
    df        = spark.createDataFrame(rows, schema=RESULTS_SCHEMA)
    df.write.mode("append").partitionBy("business_date").saveAsTable(
        KPI_VALIDATION_RESULTS
    )
    print(f"  Written {len(results)} result records.")
    return True


def write_validation_run(run_record: dict) -> bool:
    col_order = [f.name for f in RUN_SCHEMA.fields]
    rows      = [tuple(run_record.get(col) for col in col_order)]
    df        = spark.createDataFrame(rows, schema=RUN_SCHEMA)
    df.write.mode("append").partitionBy("business_date").saveAsTable(
        KPI_VALIDATION_RUN
    )
    print(f"  Written run record.")
    return True


def determine_overall_status(results: list) -> str:
    worst = STATUS_PASS
    for r in results:
        s = r.get("check_status", STATUS_PASS)
        if SEVERITY_RANK.get(s, 0) > SEVERITY_RANK.get(worst, 0):
            worst = s
    return worst


print("Write functions loaded.")

## Main Validation Loop

In [0]:
# Load all distinct Gold tables from KPI registry
kpi_rows = (
    spark.table(KPI_REGISTRY)
         .filter(F.col("is_active") == True)
         .collect()
)

if not kpi_rows:
    print("No KPIs registered. Nothing to validate.")
else:
    # Group KPIs by Gold table
    tables = {}
    for kpi in kpi_rows:
        gold_fqn = kpi["gold_fully_qualified_name"]
        if gold_fqn not in tables:
            tables[gold_fqn] = []
        tables[gold_fqn].append(kpi.asDict())

    # Filter if gold_table_filter specified
    if GOLD_TABLE_FILTER:
        tables = {
            k: v for k, v in tables.items()
            if GOLD_TABLE_FILTER in k
        }

    all_run_records    = []
    all_result_records = []

    print(f"Validating {len(tables)} Gold table(s) for {BUSINESS_DATE}\n")

    for gold_fqn, kpis in tables.items():

        TABLE_RUN_ID   = str(uuid.uuid4())
        table_name     = gold_fqn.split(".")[-1]
        table_results  = []

        # Determine business date column from first KPI
        biz_date_col = kpis[0].get("business_date_column", "business_date")

        print(f"\n{'='*60}")
        print(f"Gold table     : {table_name}")
        print(f"Run ID         : {TABLE_RUN_ID}")
        print(f"KPIs to check  : {len(kpis)}")
        print(f"{'='*60}")

        # Load Gold data for this business date once
        gold_df = load_gold_data_for_date(gold_fqn, biz_date_col, BUSINESS_DATE)
        row_count = gold_df.count()
        print(f"Gold rows for {BUSINESS_DATE}: {row_count}")

        if row_count == 0:
            print(f"No Gold data found for {BUSINESS_DATE}. Skipping.")
            continue

        # Run checks for each KPI
        for kpi in kpis:
            kpi_name   = kpi["kpi_name"]
            kpi_column = kpi["kpi_column"]
            checks     = kpi.get("checks_enabled", "[]")

            print(f"\n  KPI: {kpi_name} (column: {kpi_column})")
            print(f"  Checks: {checks}")

            # FORMULA check
            if "FORMULA" in checks and kpi.get("formula_expression"):
                result = check_kpi_formula(
                    gold_df, kpi, TABLE_RUN_ID, gold_fqn, BUSINESS_DATE
                )
                if result:
                    table_results.append(result)

            # CROSS_KPI check
            if "CROSS_KPI" in checks and kpi.get("formula_expression"):
                result = check_cross_kpi(
                    gold_df, kpi, TABLE_RUN_ID, gold_fqn, BUSINESS_DATE
                )
                if result:
                    table_results.append(result)

            # ANOMALY check
            if "ANOMALY" in checks:
                baseline = load_baseline(gold_fqn, kpi_name, kpi_column)
                result   = check_kpi_anomaly(
                    gold_df, kpi, baseline, TABLE_RUN_ID, gold_fqn, BUSINESS_DATE
                )
                if result:
                    table_results.append(result)

            # RANGE check
            if "RANGE" in checks:
                result = check_kpi_range(
                    gold_df, kpi, TABLE_RUN_ID, gold_fqn, BUSINESS_DATE
                )
                if result:
                    table_results.append(result)

        # Score overall
        overall_status  = determine_overall_status(table_results)
        failed_checks   = [
            r for r in table_results
            if r["check_status"] in (STATUS_WARNING, STATUS_CRITICAL)
        ]
        critical_checks = [
            r for r in table_results
            if r["check_status"] == STATUS_CRITICAL
        ]

        ai_status = (
            "PENDING"
            if (len(failed_checks) >= AI_MIN_FAILED_CHECKS)
            else "NOT_REQUIRED"
        )

        kpis_passed   = sum(1 for r in table_results if r["check_status"] == STATUS_PASS)
        kpis_warned   = sum(1 for r in table_results if r["check_status"] == STATUS_WARNING)
        kpis_critical = sum(1 for r in table_results if r["check_status"] == STATUS_CRITICAL)

        run_end_time = datetime.now()
        duration     = (run_end_time - RUN_START_TIME).total_seconds()

        run_record = {
            "validation_run_id"           : TABLE_RUN_ID,
            "gold_fully_qualified_name"    : gold_fqn,
            "business_date"               : BUSINESS_DATE,
            "triggered_by"                : "MANUAL",
            "agent2_reconciliation_run_id" : None,
            "validation_start_timestamp"  : RUN_START_TIME,
            "validation_end_timestamp"    : run_end_time,
            "validation_duration_seconds" : duration,
            "overall_status"              : overall_status,
            "total_kpis_checked"          : len(table_results),
            "total_kpis_passed"           : kpis_passed,
            "total_kpis_warned"           : kpis_warned,
            "total_kpis_critical"         : kpis_critical,
            "ai_enrichment_status"        : ai_status,
            "ai_enrichment_timestamp"     : None,
            "alert_sent"                  : None,
            "alert_sent_timestamp"        : None,
            "alert_channel"               : None,
            "created_timestamp"           : datetime.now(),
            "created_by"                  : "02_kpi_validation_agent"
        }

        all_run_records.append(run_record)
        all_result_records.extend(table_results)

        print(f"\n  {'─'*50}")
        print(f"  RESULTS: {table_name}")
        print(f"  {'─'*50}")
        print(f"  Overall status : {overall_status}")
        print(f"  Checks run     : {len(table_results)}")
        print(f"  Passed         : {kpis_passed}")
        print(f"  Warning        : {kpis_warned}")
        print(f"  Critical       : {kpis_critical}")
        print(f"  AI status      : {ai_status}")
        print(f"  {'─'*50}")

    # Write all results
    print("\nWriting results...")
    for run_record in all_run_records:
        write_validation_run(run_record)
    write_validation_results(all_result_records)
    print("Write complete.")

## Results Viewer

In [0]:
print("=" * 60)
print(f"AGENT 4 RESULTS — {BUSINESS_DATE}")
print("=" * 60)

# Run summary
print("\n1. Validation Run Summary:\n")
spark.sql(f"""
    SELECT
        SUBSTRING_INDEX(gold_fully_qualified_name, '.', -1) AS gold_table,
        business_date,
        overall_status,
        total_kpis_checked,
        total_kpis_passed,
        total_kpis_warned,
        total_kpis_critical,
        ai_enrichment_status,
        ROUND(validation_duration_seconds, 2)               AS duration_sec
    FROM {KPI_VALIDATION_RUN}
    WHERE business_date = '{BUSINESS_DATE}'
    ORDER BY gold_fully_qualified_name
""").show(truncate=False)

# Failed checks
print("\n2. Failed KPI Checks:\n")
spark.sql(f"""
    SELECT
        SUBSTRING_INDEX(gold_fully_qualified_name, '.', -1) AS gold_table,
        kpi_name,
        kpi_column,
        check_type,
        check_status,
        ROUND(observed_value, 4)    AS observed_value,
        ROUND(expected_value, 4)    AS expected_value,
        ROUND(deviation_percentage, 2) AS deviation_pct,
        check_detail
    FROM {KPI_VALIDATION_RESULTS}
    WHERE business_date = '{BUSINESS_DATE}'
      AND check_status IN ('WARNING', 'CRITICAL')
    ORDER BY
        gold_fully_qualified_name,
        CASE check_status WHEN 'CRITICAL' THEN 1 ELSE 2 END,
        kpi_name
""").show(50, truncate=False)

## Final Summary

In [0]:
print("\n" + "=" * 60)
print("AGENT 4 KPI VALIDATION COMPLETE")
print("=" * 60)
print(f"Business date  : {BUSINESS_DATE}")
print(f"Tables checked : {len(all_run_records)}")
print(f"Total KPI checks: {len(all_result_records)}")
print(f"Completed at   : {datetime.now()}")
print()

for run in all_run_records:
    table  = run["gold_fully_qualified_name"].split(".")[-1]
    status = run["overall_status"]
    emoji  = "✅" if status == "PASS" else "🔴" if status == "CRITICAL" else "🟡"
    print(
        f"  {emoji} {table:<45} "
        f"Status: {status}  "
        f"Checks: {run['total_kpis_checked']} run  "
        f"AI: {run['ai_enrichment_status']}"
    )

print("=" * 60)